5.1回来，不知道哪根筋搭错了，突然对自动调参有了想法。一顿狂暴模式，花了4天写了个beta版出来，发布给算法用。又花了一周不断完善，想法越来越多，即使有code agent辅助，也还是需要斟酌各个想法的优先级。  
今天5.18，读了一篇ai新闻报道，说的是xAI收购cursor的事，指出100亿美金意在获取cursor的用户交互数据对模型做微调。这再次触发我新想法：这个自动调参agent，和code agent有什么区别呢？也是可以收集算法不断的调参尝试作为样本，对某个LLM模型进行微调。  
也许这想法来的有点晚：训个模型，是我从开始做agent相关，就不断有人在提的事。但我不能确定，提这些的人是否真的理解了这个模型该怎么训？反正我之前是不太理解的，现在也只是有了想法。  
anyway，沿着这个想法一路想，我觉得auto-tune-agent不是一个简单的玩具，它是一个需要长时间建设才可能看到效果的东西。我不确定身边的人及leader是否get到这点。目前有多个类似项目在进行，且目标都是要"做出业务效果"，也就是能通过agent提升某个实验的指标，以此来证明价值。以公司及领导的耐心，可能3个月就必须出结果。如果到时候没有效果，我不清楚这个项目是否还存在，是否能继续下去。而我的判断，agent出效果，还需要大量的工作。比如：eval体系建设——没有评估体系怎么定优化方向呢，context组织优化——如何组织context能获取更好结果，搜参方式——除了让模型做决策暴力搜之外还可以辅以离线搜参提升效率，更需要长期积累的：用户用的越多，调参数据就越多，这些数据也许可以微调一个模型，像for code的微调一样，提升调参效率，这个更是需要长期的数据积累。  
基于以上，要做一个真正有效的auto-tune-agent，可能需要几年的建设。如果在这里无法实现，那我只能换地方继续这个工作了。所以我需要把中间的思考记下来，方便未来迫不得已迁移。  
有意思的是，目前这个时代，已经没有必要"带着代码跑路"了。我只需要把思考的过程记下来，不带任何细节，就可以换一个地方重建，而且还方便因地制宜。所以我更得多记录了。

agent自动调参，后面统称为auto-tune-agent，和claude code/opencode/codex有什么区别？开始做agent应用相关时候，我以为所谓"agent xxx"就是基于某个code agent，如以上所述这些，将agent可以调用的接口通过tools/MCP/CLI等方式接入agent，然后再编写一些skill约束流程，然后再完善相关的知识库，即可。在这个架构里，code agent是核心入口，tools/MCP/CLI等是agent手脚，skill控制agent流程，整个流程入口就是聊天框输入。  
后来我发现，这样有很大局限性。入口是聊天框，也就是说交互模式就是“输入-输出”，一进一出。但想想auto-tune-agent这个场景，它的过程实际上是个loop，而不是"输入-输出"这种模式。auto-tune-agent的一个loop包含：LLM给建议->根据建议搜参->应用参数->等待ab结果->收集指标->评估结果指标和建议指标差异->归因/总结经验。之后是下一次循环，直到实验到达一定轮数或实验目标达成。在这个loop中，LLM的角色是其中一环——用来根据输入给建议，LLM不控制任何流程，全靠loop代码控制，LLM只负责在固定环节做决策。  
一旦你认识到，整个流程是个loop，而不是code agent中的“输入输出”模式，就发现，其实所有的所谓"agent"，都是个loop。包括claude code/opencode/codex，本质也是在LLM上包了一层代码做loop。这层代码针对coding环境，进行了一系列针对性的优化，如流程控制、session实现、context控制，等等，最终让LLM可以在coding这个场景下发挥作用。这么看，auto-tune-agent不应该只是code agent中的一个skill，而应该是一个平行关系的独立agent，有自己的loop循环、context控制、memory记录等等。也就是说：我们要建的是一个自动调参场景的claude code。  
再延伸一步，任何一个场景，想让LLM发挥作用，都需要一个代码来控制流程，LLM只是流程中负责某个决策的一环。这么看，有太多场景可以通过LLM去重塑了。